# Adventure Works — Data Exploration & Profiling

This notebook performs the initial exploration and profiling of the Adventure Works source datasets.

Objectives:
- Load all raw CSV datasets
- Inspect their structure and schemas
- Calculate row and column counts
- Identify missing and duplicate values
- Understand relationships between datasets
- Identify potential primary and foreign keys
- Establish a foundation for Bronze-layer ingestion and downstream transformations

In [2]:
#Importing Libraries
import pandas as pd
from pathlib import Path
import os

print("Pandas version:",pd.__version__)
print("Working directory:",Path.cwd())

Pandas version: 3.0.1
Working directory: a:\DataEngineering\DeProjects\AdventureWorksProject\Adventure-Works-Azure-Data-Engineering\notebooks


In [3]:
#Locating the raw data folder
# Project root = one level above the notebooks folder
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)
print("Raw folder exists:", RAW_DATA_PATH.exists())

Project root: a:\DataEngineering\DeProjects\AdventureWorksProject\Adventure-Works-Azure-Data-Engineering
Raw data path: a:\DataEngineering\DeProjects\AdventureWorksProject\Adventure-Works-Azure-Data-Engineering\data\raw
Raw folder exists: True


In [4]:
#Listing all CSVs
csv_files = sorted(RAW_DATA_PATH.glob("*.csv"))
print(f"Total CSV files found(count):{len(csv_files)}")
for file in csv_files:
    print(file.name)

Total CSV files found(count):10
AdventureWorks_Calendar.csv
AdventureWorks_Customers.csv
AdventureWorks_Product_Categories.csv
AdventureWorks_Product_Subcategories.csv
AdventureWorks_Products.csv
AdventureWorks_Returns.csv
AdventureWorks_Sales_2015.csv
AdventureWorks_Sales_2016.csv
AdventureWorks_Sales_2017.csv
AdventureWorks_Territories.csv


In [5]:
# Loading all datasets into a dictionary of dataframes

datasets = {}

for file in csv_files:
    dataset_name = file.stem.replace("AdventureWorks_", "")

    try:
        df = pd.read_csv(file, encoding="utf-8")
    except UnicodeDecodeError:
        print(f"UTF-8 failed for {file.name}. Trying cp1252...")
        df = pd.read_csv(file, encoding="cp1252")

    datasets[dataset_name] = df

    print(
        f"{dataset_name} dataset loaded with shape: {df.shape} "
        f"Rows: {len(df)}, "
        f"Columns: {len(df.columns)} "
        f"Columns: {df.columns.tolist()}"
    )

Calendar dataset loaded with shape: (912, 1) Rows: 912, Columns: 1 Columns: ['Date']
UTF-8 failed for AdventureWorks_Customers.csv. Trying cp1252...
Customers dataset loaded with shape: (18148, 13) Rows: 18148, Columns: 13 Columns: ['CustomerKey', 'Prefix', 'FirstName', 'LastName', 'BirthDate', 'MaritalStatus', 'Gender', 'EmailAddress', 'AnnualIncome', 'TotalChildren', 'EducationLevel', 'Occupation', 'HomeOwner']
Product_Categories dataset loaded with shape: (4, 2) Rows: 4, Columns: 2 Columns: ['ProductCategoryKey', 'CategoryName']
Product_Subcategories dataset loaded with shape: (37, 3) Rows: 37, Columns: 3 Columns: ['ProductSubcategoryKey', 'SubcategoryName', 'ProductCategoryKey']
Products dataset loaded with shape: (293, 11) Rows: 293, Columns: 11 Columns: ['ProductKey', 'ProductSubcategoryKey', 'ProductSKU', 'ProductName', 'ModelName', 'ProductDescription', 'ProductColor', 'ProductSize', 'ProductStyle', 'ProductCost', 'ProductPrice']
Returns dataset loaded with shape: (1809, 4) Row

In [6]:
#Creating dataset summary
summary = []

for name, df in datasets.items():
    summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Duplicate Rows": df.duplicated().sum(),
        "Total Null Values": df.isna().sum().sum()
    })

dataset_summary = pd.DataFrame(summary)

dataset_summary

,Dataset,Rows,Columns,Duplicate Rows,Total Null Values
0,Calendar,912,1,0,0
1,Customers,18148,13,0,260
2,Product_Categories,4,2,0,0
3,Product_Subcategories,37,3,0,0
4,Products,293,11,0,50
5,Returns,1809,4,0,0
6,Sales_2015,2630,8,0,0
7,Sales_2016,23935,8,0,0
8,Sales_2017,29481,8,0,0
9,Territories,10,4,0,0


In [7]:
#Inspecting Schema
for name, df in datasets.items():
    print("=" * 80)
    print(f"DATASET: {name}")
    print("=" * 80)
    df.info()
    print()

DATASET: Calendar
<class 'pandas.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Date    912 non-null    str  
dtypes: str(1)
memory usage: 7.3 KB

DATASET: Customers


<class 'pandas.DataFrame'>
RangeIndex: 18148 entries, 0 to 18147
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   CustomerKey     18148 non-null  int64
 1   Prefix          18018 non-null  str  
 2   FirstName       18148 non-null  str  
 3   LastName        18148 non-null  str  
 4   BirthDate       18148 non-null  str  
 5   MaritalStatus   18148 non-null  str  
 6   Gender          18018 non-null  str  
 7   EmailAddress    18148 non-null  str  
 8   AnnualIncome    18148 non-null  str  
 9   TotalChildren   18148 non-null  int64
 10  EducationLevel  18148 non-null  str  
 11  Occupation      18148 non-null  str  
 12  HomeOwner       18148 non-null  str  
dtypes: int64(2), str(11)
memory usage: 1.8 MB

DATASET: Product_Categories
<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  ----

In [8]:
#Displaying column names
for name, df in datasets.items():
    print(f"\n{name}")
    print(list(df.columns))


Calendar
['Date']

Customers
['CustomerKey', 'Prefix', 'FirstName', 'LastName', 'BirthDate', 'MaritalStatus', 'Gender', 'EmailAddress', 'AnnualIncome', 'TotalChildren', 'EducationLevel', 'Occupation', 'HomeOwner']

Product_Categories
['ProductCategoryKey', 'CategoryName']

Product_Subcategories
['ProductSubcategoryKey', 'SubcategoryName', 'ProductCategoryKey']

Products
['ProductKey', 'ProductSubcategoryKey', 'ProductSKU', 'ProductName', 'ModelName', 'ProductDescription', 'ProductColor', 'ProductSize', 'ProductStyle', 'ProductCost', 'ProductPrice']

Returns
['ReturnDate', 'TerritoryKey', 'ProductKey', 'ReturnQuantity']

Sales_2015
['OrderDate', 'StockDate', 'OrderNumber', 'ProductKey', 'CustomerKey', 'TerritoryKey', 'OrderLineItem', 'OrderQuantity']

Sales_2016
['OrderDate', 'StockDate', 'OrderNumber', 'ProductKey', 'CustomerKey', 'TerritoryKey', 'OrderLineItem', 'OrderQuantity']

Sales_2017
['OrderDate', 'StockDate', 'OrderNumber', 'ProductKey', 'CustomerKey', 'TerritoryKey', 'OrderL

In [9]:
#Checking sample records
for name, df in datasets.items():
    print("=" * 80)
    print(f"{name} — First 5 Records")
    print("=" * 80)
    display(df.head())

Calendar — First 5 Records


,Date
0,1/1/2015
1,1/2/2015
2,1/3/2015
3,1/4/2015
4,1/5/2015


Customers — First 5 Records


,CustomerKey,Prefix,FirstName,LastName,BirthDate,MaritalStatus,Gender,EmailAddress,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner
0,11000,MR.,JON,YANG,4/8/1966,M,M,jon24@adventure-works.com,"$90,000",2,Bachelors,Professional,Y
1,11001,MR.,EUGENE,HUANG,5/14/1965,S,M,eugene10@adventure-works.com,"$60,000",3,Bachelors,Professional,N
2,11002,MR.,RUBEN,TORRES,8/12/1965,M,M,ruben35@adventure-works.com,"$60,000",3,Bachelors,Professional,Y
3,11003,MS.,CHRISTY,ZHU,2/15/1968,S,F,christy12@adventure-works.com,"$70,000",0,Bachelors,Professional,N
4,11004,MRS.,ELIZABETH,JOHNSON,8/8/1968,S,F,elizabeth5@adventure-works.com,"$80,000",5,Bachelors,Professional,Y


Product_Categories — First 5 Records


...

,OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
0,1/1/2016,10/17/2002,SO48797,385,14335,1,1,1
1,1/1/2016,9/30/2002,SO48802,383,24923,9,1,1
2,1/1/2016,11/29/2002,SO48801,326,15493,1,1,1
3,1/1/2016,11/16/2002,SO48799,352,26708,4,1,1
4,1/1/2016,12/16/2002,SO48798,369,23332,9,1,1


Sales_2017 — First 5 Records


,OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
0,1/1/2017,12/13/2003,SO61285,529,23791,1,2,2
1,1/1/2017,9/24/2003,SO61285,214,23791,1,3,1
2,1/1/2017,9/4/2003,SO61285,540,23791,1,1,1
3,1/1/2017,9/28/2003,SO61301,529,16747,1,2,2
4,1/1/2017,10/21/2003,SO61301,377,16747,1,1,1


Territories — First 5 Records


,SalesTerritoryKey,Region,Country,Continent
0,1,Northwest,United States,North America
1,2,Northeast,United States,North America
2,3,Central,United States,North America
3,4,Southwest,United States,North America
4,5,Southeast,United States,North America


In [10]:
#Null value analysis
null_summary = {}

for name, df in datasets.items():
    null_counts = df.isna().sum()
    null_counts = null_counts[null_counts > 0]
    null_summary[name] = null_counts

for name, null_counts in null_summary.items():
    print(f"\n{name}")
    
    if len(null_counts) == 0:
        print("No null values found.")
    else:
        print(null_counts)


Calendar
No null values found.

Customers
Prefix    130
Gender    130
dtype: int64

Product_Categories
No null values found.

Product_Subcategories
No null values found.

Products
ProductColor    50
dtype: int64

Returns
No null values found.

Sales_2015
No null values found.

Sales_2016
No null values found.

Sales_2017
No null values found.

Territories
No null values found.


In [11]:
#Duplicate analysis 
duplicate_summary = []

for name, df in datasets.items():
    duplicate_summary.append({
        "Dataset": name,
        "Duplicate Rows": df.duplicated().sum()
    })

pd.DataFrame(duplicate_summary)

,Dataset,Duplicate Rows
0,Calendar,0
1,Customers,0
2,Product_Categories,0
3,Product_Subcategories,0
4,Products,0
5,Returns,0
6,Sales_2015,0
7,Sales_2016,0
8,Sales_2017,0
9,Territories,0


In [12]:
#Primary key validation
key_columns = {
    "Customers": "CustomerKey",
    "Product_Categories": "ProductCategoryKey",
    "Product_Subcategories": "ProductSubcategoryKey",
    "Products": "ProductKey",
    "Territories": "SalesTerritoryKey",
}

for dataset, column in key_columns.items():
    df = datasets[dataset]

    print("=" * 70)
    print(f"{dataset} -> {column}")
    print(f"Total rows: {len(df):,}")
    print(f"Unique values: {df[column].nunique():,}")
    print(f"Null values: {df[column].isna().sum():,}")
    print(f"Duplicate key values: {df[column].duplicated().sum():,}")

Customers -> CustomerKey
Total rows: 18,148
Unique values: 18,148
Null values: 0
Duplicate key values: 0
Product_Categories -> ProductCategoryKey
Total rows: 4
Unique values: 4
Null values: 0
Duplicate key values: 0
Product_Subcategories -> ProductSubcategoryKey
Total rows: 37
Unique values: 37
Null values: 0
Duplicate key values: 0
Products -> ProductKey
Total rows: 293
Unique values: 293
Null values: 0
Duplicate key values: 0
Territories -> SalesTerritoryKey
Total rows: 10
Unique values: 10
Null values: 0
Duplicate key values: 0


In [13]:
#Sales key investigation
sales_columns = [
    "OrderNumber",
    "OrderLineItem"
]

for year in ["Sales_2015", "Sales_2016", "Sales_2017"]:
    df = datasets[year]

    duplicates = df.duplicated(
        subset=sales_columns
    ).sum()

    print(f"{year}: duplicate OrderNumber + OrderLineItem = {duplicates}")

Sales_2015: duplicate OrderNumber + OrderLineItem = 0
Sales_2016: duplicate OrderNumber + OrderLineItem = 0
Sales_2017: duplicate OrderNumber + OrderLineItem = 0


In [14]:
#Checking all foregin key relationships
relationships = [
    ("Sales_2015", "ProductKey", "Products", "ProductKey"),
    ("Sales_2015", "CustomerKey", "Customers", "CustomerKey"),
    ("Sales_2015", "TerritoryKey", "Territories", "SalesTerritoryKey"),

    ("Sales_2016", "ProductKey", "Products", "ProductKey"),
    ("Sales_2016", "CustomerKey", "Customers", "CustomerKey"),
    ("Sales_2016", "TerritoryKey", "Territories", "SalesTerritoryKey"),

    ("Sales_2017", "ProductKey", "Products", "ProductKey"),
    ("Sales_2017", "CustomerKey", "Customers", "CustomerKey"),
    ("Sales_2017", "TerritoryKey", "Territories", "SalesTerritoryKey"),

    ("Products", "ProductSubcategoryKey",
     "Product_Subcategories", "ProductSubcategoryKey"),

    ("Product_Subcategories", "ProductCategoryKey",
     "Product_Categories", "ProductCategoryKey"),
]

for child_ds, child_col, parent_ds, parent_col in relationships:

    child_values = set(datasets[child_ds][child_col].dropna().unique())
    parent_values = set(datasets[parent_ds][parent_col].dropna().unique())

    missing = child_values - parent_values

    print(
        f"{child_ds}.{child_col} → "
        f"{parent_ds}.{parent_col}: "
        f"{len(missing)} unmatched values"
    )

Sales_2015.ProductKey → Products.ProductKey: 0 unmatched values
Sales_2015.CustomerKey → Customers.CustomerKey: 0 unmatched values
Sales_2015.TerritoryKey → Territories.SalesTerritoryKey: 0 unmatched values
Sales_2016.ProductKey → Products.ProductKey: 0 unmatched values
Sales_2016.CustomerKey → Customers.CustomerKey: 0 unmatched values
Sales_2016.TerritoryKey → Territories.SalesTerritoryKey: 0 unmatched values
Sales_2017.ProductKey → Products.ProductKey: 0 unmatched values
Sales_2017.CustomerKey → Customers.CustomerKey: 0 unmatched values
Sales_2017.TerritoryKey → Territories.SalesTerritoryKey: 0 unmatched values
Products.ProductSubcategoryKey → Product_Subcategories.ProductSubcategoryKey: 0 unmatched values
Product_Subcategories.ProductCategoryKey → Product_Categories.ProductCategoryKey: 0 unmatched values


In [15]:
#Checking date ranges
date_columns = {
    "Calendar": "Date",
    "Customers": "BirthDate",
    "Returns": "ReturnDate",
    "Sales_2015": "OrderDate",
    "Sales_2016": "OrderDate",
    "Sales_2017": "OrderDate",
}

for dataset, column in date_columns.items():

    df = datasets[dataset].copy()
    dates = pd.to_datetime(df[column], errors="coerce")

    print("=" * 70)
    print(dataset)
    print("Invalid dates:", dates.isna().sum())
    print("Minimum date:", dates.min())
    print("Maximum date:", dates.max())

Calendar
Invalid dates: 0
Minimum date: 2015-01-01 00:00:00
Maximum date: 2017-06-30 00:00:00
Customers
Invalid dates: 0
Minimum date: 1910-08-13 00:00:00
Maximum date: 1980-12-26 00:00:00
Returns
Invalid dates: 0
Minimum date: 2015-01-18 00:00:00
Maximum date: 2017-06-30 00:00:00
Sales_2015
Invalid dates: 0
Minimum date: 2015-01-01 00:00:00
Maximum date: 2015-12-31 00:00:00
Sales_2016
Invalid dates: 0
Minimum date: 2016-01-01 00:00:00
Maximum date: 2016-12-31 00:00:00
Sales_2017
Invalid dates: 0
Minimum date: 2017-01-01 00:00:00
Maximum date: 2017-06-30 00:00:00


In [16]:
# Convert AnnualIncome from string to numeric
datasets["Customers"]["AnnualIncome"] = (
    datasets["Customers"]["AnnualIncome"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

In [17]:
#Checking numerical changes
numeric_columns = {
    "Customers": ["AnnualIncome", "TotalChildren"],
    "Products": ["ProductCost", "ProductPrice"],
    "Returns": ["ReturnQuantity"],
    "Sales_2015": ["OrderQuantity"],
    "Sales_2016": ["OrderQuantity"],
    "Sales_2017": ["OrderQuantity"],
}

for dataset, columns in numeric_columns.items():

    print("=" * 70)
    print(dataset)

    for column in columns:
        print(f"\n{column}")
        print("Min:", datasets[dataset][column].min())
        print("Max:", datasets[dataset][column].max())
        print("Negative values:",
              (datasets[dataset][column] < 0).sum())

Customers

AnnualIncome
Min: 10000.0
Max: 170000.0
Negative values: 0

TotalChildren
Min: 0
Max: 5
Negative values: 0
Products

ProductCost
Min: 0.8565
Max: 2171.2942
Negative values: 0

ProductPrice
Min: 2.29
Max: 3578.27
Negative values: 0
Returns

ReturnQuantity
Min: 1
Max: 2
Negative values: 0
Sales_2015

OrderQuantity
Min: 1
Max: 1
Negative values: 0
Sales_2016

OrderQuantity
Min: 1
Max: 3
Negative values: 0
Sales_2017

OrderQuantity
Min: 1
Max: 3
Negative values: 0


In [18]:
for dataset, columns in numeric_columns.items():
    print("=" * 70)
    print(dataset)

    for column in columns:
        print(column, "->", datasets[dataset][column].dtype)

Customers
AnnualIncome -> float64
TotalChildren -> int64
Products
ProductCost -> float64
ProductPrice -> float64
Returns
ReturnQuantity -> int64
Sales_2015
OrderQuantity -> int64
Sales_2016
OrderQuantity -> int64
Sales_2017
OrderQuantity -> int64


In [19]:
#Profile categorical columns
categorical_columns = {
    "Customers": ["Gender", "MaritalStatus", "EducationLevel", "Occupation"],
    "Products": ["ProductColor", "ProductSize", "ProductStyle"],
    "Territories": ["Region", "Country", "Continent"],
}

for dataset, columns in categorical_columns.items():

    print("=" * 70)
    print(dataset)

    for column in columns:
        print(f"\n{column}")
        print(datasets[dataset][column].value_counts(dropna=False).head(15))

Customers

Gender
Gender
M      9126
F      8892
NaN     130
Name: count, dtype: int64

MaritalStatus
MaritalStatus
M    9817
S    8331
Name: count, dtype: int64

EducationLevel
EducationLevel
Bachelors              5261
Partial College        4966
High School            3241
Graduate Degree        3125
Partial High School    1555
Name: count, dtype: int64

Occupation
Occupation
Professional      5424
Skilled Manual    4501
Management        3011
Clerical          2859
Manual            2353
Name: count, dtype: int64
Products

ProductColor
ProductColor
Black           88
NaN             50
Red             37
Silver          36
Yellow          36
Blue            26
Multi            8
Silver/Black     7
White            4
Grey             1
Name: count, dtype: int64

ProductSize
ProductSize
0     84
44    29
48    25
52    16
42    15
58    13
38    12
M     11
L     11
62    11
60    11
46    11
40    11
S      9
50     9
Name: count, dtype: int64

ProductStyle
ProductStyle
U    174
0  

In [20]:
#Combining the three Sales datasets for profiling
sales = pd.concat(
    [
        datasets["Sales_2015"],
        datasets["Sales_2016"],
        datasets["Sales_2017"]
    ],
    ignore_index=True
)

print("Combined Sales Shape:", sales.shape)
print("Sales rows:", len(sales))

Combined Sales Shape: (56046, 8)
Sales rows: 56046


In [21]:
sales["OrderDate"] = pd.to_datetime(
    sales["OrderDate"],
    errors="coerce"
)

print("Sales date range:")
print("Min:", sales["OrderDate"].min())
print("Max:", sales["OrderDate"].max())

Sales date range:
Min: 2015-01-01 00:00:00
Max: 2017-06-30 00:00:00


In [22]:
#Final profiling report
profile_report = []

for name, df in datasets.items():

    for column in df.columns:
        profile_report.append({
            "Dataset": name,
            "Column": column,
            "Data Type": str(df[column].dtype),
            "Rows": len(df),
            "Null Count": df[column].isna().sum(),
            "Null %": round(df[column].isna().mean() * 100, 2),
            "Unique Values": df[column].nunique(),
            "Duplicate Values": df[column].duplicated().sum()
        })

profile_report_df = pd.DataFrame(profile_report)

profile_report_df

,Dataset,Column,Data Type,Rows,Null Count,Null %,Unique Values,Duplicate Values
0,Calendar,Date,str,912,0,0.00,912,0
1,Customers,CustomerKey,int64,18148,0,0.00,18148,0
2,Customers,Prefix,str,18148,130,0.72,3,18144
3,Customers,FirstName,str,18148,0,0.00,666,17482
4,Customers,LastName,str,18148,0,0.00,372,17776
...,...,...,...,...,...,...,...,...
57,Sales_2017,OrderQuantity,int64,29481,0,0.00,3,29478
58,Territories,SalesTerritoryKey,int64,10,0,0.00,10,0
59,Territories,Region,str,10,0,0.00,10,0
60,Territories,Country,str,10,0,0.00,6,4


In [23]:
#Verifying returns relationship
return_relationships = [
    ("Returns", "ProductKey", "Products", "ProductKey"),
    ("Returns", "TerritoryKey", "Territories", "SalesTerritoryKey"),
]

for child_ds, child_col, parent_ds, parent_col in return_relationships:

    child_values = set(datasets[child_ds][child_col].dropna().unique())
    parent_values = set(datasets[parent_ds][parent_col].dropna().unique())

    missing = child_values - parent_values

    print(
        f"{child_ds}.{child_col} → "
        f"{parent_ds}.{parent_col}: "
        f"{len(missing)} unmatched values"
    )

Returns.ProductKey → Products.ProductKey: 0 unmatched values
Returns.TerritoryKey → Territories.SalesTerritoryKey: 0 unmatched values


In [24]:
#Checking the Product hierarchy completely
print("Products:", datasets["Products"]["ProductKey"].nunique())
print(
    "Products with missing subcategory:",
    datasets["Products"]["ProductSubcategoryKey"].isna().sum()
)

print(
    "Subcategories with missing category:",
    datasets["Product_Subcategories"]["ProductCategoryKey"].isna().sum()
)

Products: 293
Products with missing subcategory: 0
Subcategories with missing category: 0


In [25]:
#Validating the sales file boundaries
for year in ["Sales_2015", "Sales_2016", "Sales_2017"]:
    df = datasets[year].copy()
    df["OrderDate"] = pd.to_datetime(df["OrderDate"])

    print(
        year,
        "| Min:", df["OrderDate"].min(),
        "| Max:", df["OrderDate"].max(),
        "| Rows:", len(df)
    )

Sales_2015 | Min: 2015-01-01 00:00:00 | Max: 2015-12-31 00:00:00 | Rows: 2630
Sales_2016 | Min: 2016-01-01 00:00:00 | Max: 2016-12-31 00:00:00 | Rows: 23935
Sales_2017 | Min: 2017-01-01 00:00:00 | Max: 2017-06-30 00:00:00 | Rows: 29481


In [26]:
#Checking buisness-rule anomalies
checks = {
    "Negative OrderQuantity 2015":
        (datasets["Sales_2015"]["OrderQuantity"] < 0).sum(),

    "Negative OrderQuantity 2016":
        (datasets["Sales_2016"]["OrderQuantity"] < 0).sum(),

    "Negative OrderQuantity 2017":
        (datasets["Sales_2017"]["OrderQuantity"] < 0).sum(),

    "Negative ReturnQuantity":
        (datasets["Returns"]["ReturnQuantity"] < 0).sum(),

    "Negative ProductCost":
        (datasets["Products"]["ProductCost"] < 0).sum(),

    "Negative ProductPrice":
        (datasets["Products"]["ProductPrice"] < 0).sum(),

    "Negative AnnualIncome":
        (datasets["Customers"]["AnnualIncome"] < 0).sum(),
}

for check, count in checks.items():
    print(f"{check}: {count}")

Negative OrderQuantity 2015: 0
Negative OrderQuantity 2016: 0
Negative OrderQuantity 2017: 0
Negative ReturnQuantity: 0
Negative ProductCost: 0
Negative ProductPrice: 0
Negative AnnualIncome: 0


In [27]:
#Checking logical product pricing
products = datasets["Products"]

print(
    "Products where ProductPrice < ProductCost:",
    (products["ProductPrice"] < products["ProductCost"]).sum()
)

Products where ProductPrice < ProductCost: 0


In [28]:
#Checking sales grain globally
sales = pd.concat(
    [
        datasets["Sales_2015"],
        datasets["Sales_2016"],
        datasets["Sales_2017"]
    ],
    ignore_index=True
)

duplicates = sales.duplicated(
    subset=["OrderNumber", "OrderLineItem"]
).sum()

print("Combined Sales rows:", len(sales))
print("Duplicate OrderNumber + OrderLineItem:", duplicates)

Combined Sales rows: 56046
Duplicate OrderNumber + OrderLineItem: 0


In [29]:
#Checking OrderNumber vs line-item grain 
orders = sales.groupby("OrderNumber").size()

print("Total unique orders:", orders.index.nunique())
print("Average line items per order:", orders.mean())
print("Maximum line items in one order:", orders.max())

Total unique orders: 25164
Average line items per order: 2.227229375298045
Maximum line items in one order: 8


In [30]:
# Verifying that Bronze has the same record counts as Raw

from pathlib import Path
import pandas as pd

BRONZE_PATH = PROJECT_ROOT / "data" / "bronze"

bronze_summary = []

for dataset_dir in sorted(BRONZE_PATH.iterdir()):

    if not dataset_dir.is_dir():
        continue

    csv_files = list(dataset_dir.glob("*.csv"))

    if not csv_files:
        continue

    file = csv_files[0]

    # Try common encodings
    df = None

    for encoding in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            df = pd.read_csv(file, encoding=encoding)
            print(f"{dataset_dir.name}: loaded using {encoding}")
            break
        except UnicodeDecodeError:
            continue

    if df is None:
        raise UnicodeDecodeError(
            "unknown",
            b"",
            0,
            1,
            f"Could not decode {file}"
        )

    bronze_summary.append({
        "Dataset": dataset_dir.name,
        "Rows": len(df),
        "Columns": len(df.columns)
    })

bronze_summary_df = pd.DataFrame(bronze_summary)

bronze_summary_df

Calendar: loaded using utf-8


Customers: loaded using utf-8
Product_Categories: loaded using utf-8
Product_Subcategories: loaded using utf-8
Products: loaded using utf-8
Returns: loaded using utf-8
Sales_2015: loaded using utf-8
Sales_2016: loaded using utf-8
Sales_2017: loaded using utf-8
Territories: loaded using utf-8


,Dataset,Rows,Columns
0,Calendar,912,1
1,Customers,18148,13
2,Product_Categories,4,2
3,Product_Subcategories,37,3
4,Products,293,11
5,Returns,1809,4
6,Sales_2015,2630,8
7,Sales_2016,23935,8
8,Sales_2017,29481,8
9,Territories,10,4


## Bronze Layer Validation
Validate that the Bronze layer contains the same records and columns as the Raw source data.

In [34]:
from pathlib import Path
import pandas as pd

RAW_PATH = PROJECT_ROOT / "data" / "raw"
BRONZE_PATH = PROJECT_ROOT / "data" / "bronze"


def read_csv_safe(file):
    """
    Read CSV using the first encoding that works.
    """
    for encoding in ["utf-8", "utf-8-sig", "cp1252", "latin1"]:
        try:
            df = pd.read_csv(file, encoding=encoding)
            return df, encoding
        except UnicodeDecodeError:
            continue

    raise ValueError(f"Could not decode file: {file}")


validation_results = []

raw_file_map = {
    "Calendar": "AdventureWorks_Calendar.csv",
    "Customers": "AdventureWorks_Customers.csv",
    "Product_Categories": "AdventureWorks_Product_Categories.csv",
    "Product_Subcategories": "AdventureWorks_Product_Subcategories.csv",
    "Products": "AdventureWorks_Products.csv",
    "Returns": "AdventureWorks_Returns.csv",
    "Sales_2015": "AdventureWorks_Sales_2015.csv",
    "Sales_2016": "AdventureWorks_Sales_2016.csv",
    "Sales_2017": "AdventureWorks_Sales_2017.csv",
    "Territories": "AdventureWorks_Territories.csv",
}


for dataset_name in datasets.keys():

    # ---------------------------------------------------------------
    # Raw file
    # ---------------------------------------------------------------

    raw_file = RAW_PATH / raw_file_map[dataset_name]

    # ---------------------------------------------------------------
    # Bronze file
    # ---------------------------------------------------------------

    bronze_file = (
        BRONZE_PATH
        / dataset_name
        / f"{dataset_name}.csv"
    )

    # ---------------------------------------------------------------
    # Read Raw and Bronze
    # ---------------------------------------------------------------

    raw_df, raw_encoding = read_csv_safe(raw_file)
    bronze_df, bronze_encoding = read_csv_safe(bronze_file)

    # ---------------------------------------------------------------
    # Validation
    # ---------------------------------------------------------------

    validation_results.append({
        "Dataset": dataset_name,

        "Raw Rows": len(raw_df),
        "Bronze Rows": len(bronze_df),
        "Row Count Match": len(raw_df) == len(bronze_df),

        "Raw Columns": len(raw_df.columns),
        "Bronze Columns": len(bronze_df.columns),
        "Column Count Match": (
            len(raw_df.columns) == len(bronze_df.columns)
        ),

        "Schema Match": (
            list(raw_df.columns) == list(bronze_df.columns)
        ),

        "Raw Encoding": raw_encoding,
        "Bronze Encoding": bronze_encoding,
    })


bronze_validation = pd.DataFrame(validation_results)

bronze_validation

,Dataset,Raw Rows,Bronze Rows,Row Count Match,Raw Columns,Bronze Columns,Column Count Match,Schema Match,Raw Encoding,Bronze Encoding
0,Calendar,912,912,True,1,1,True,True,utf-8,utf-8
1,Customers,18148,18148,True,13,13,True,True,cp1252,utf-8
2,Product_Categories,4,4,True,2,2,True,True,utf-8,utf-8
3,Product_Subcategories,37,37,True,3,3,True,True,utf-8,utf-8
4,Products,293,293,True,11,11,True,True,utf-8,utf-8
5,Returns,1809,1809,True,4,4,True,True,utf-8,utf-8
6,Sales_2015,2630,2630,True,8,8,True,True,utf-8,utf-8
7,Sales_2016,23935,23935,True,8,8,True,True,utf-8,utf-8
8,Sales_2017,29481,29481,True,8,8,True,True,utf-8,utf-8
9,Territories,10,10,True,4,4,True,True,utf-8,utf-8


Creating an overall Bronze validation result

In [36]:
all_checks_passed = (
    bronze_validation["Row Count Match"].all()
    and bronze_validation["Column Count Match"].all()
    and bronze_validation["Schema Match"].all()
)

print("=" * 70)
print("BRONZE VALIDATION RESULT")
print("=" * 70)

if all_checks_passed:
    print("✅ ALL BRONZE VALIDATION CHECKS PASSED")
else:
    print("❌ BRONZE VALIDATION FAILED")

print(
    f"\nDatasets validated: "
    f"{len(bronze_validation)}"
)

BRONZE VALIDATION RESULT
✅ ALL BRONZE VALIDATION CHECKS PASSED

Datasets validated: 10


Validating the ingestion log

In [37]:
LOG_FILE = PROJECT_ROOT / "logs" / "ingestion_log.csv"

ingestion_log = pd.read_csv(LOG_FILE)

print(ingestion_log)

                 dataset                                        source_file  \
0               Calendar  A:\DataEngineering\DeProjects\AdventureWorksPr...   
1              Customers  A:\DataEngineering\DeProjects\AdventureWorksPr...   
2     Product_Categories  A:\DataEngineering\DeProjects\AdventureWorksPr...   
3  Product_Subcategories  A:\DataEngineering\DeProjects\AdventureWorksPr...   
4               Products  A:\DataEngineering\DeProjects\AdventureWorksPr...   
5                Returns  A:\DataEngineering\DeProjects\AdventureWorksPr...   
6             Sales_2015  A:\DataEngineering\DeProjects\AdventureWorksPr...   
7             Sales_2016  A:\DataEngineering\DeProjects\AdventureWorksPr...   
8             Sales_2017  A:\DataEngineering\DeProjects\AdventureWorksPr...   
9            Territories  A:\DataEngineering\DeProjects\AdventureWorksPr...   

   rows_processed  columns_processed   status                  start_time  \
0             912                  1  SUCCESS  2026-0

In [38]:
print(
    "Total ingestion records:",
    len(ingestion_log)
)

print(
    "Successful:",
    (ingestion_log["status"] == "SUCCESS").sum()
)

print(
    "Failed:",
    (ingestion_log["status"] == "FAILED").sum()
)

Total ingestion records: 10
Successful: 10
Failed: 0
